# 03 Baseline Models

This notebook trains baseline regressors and saves MAE/RMSE metrics for comparison against CNN.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.evaluate import evaluate_predictions, save_metrics

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

candidate_files = [
    processed_dir / "desharnais_processed.csv",
    processed_dir / "cocomo81_processed.csv",
    processed_dir / "china_processed.csv",
]

dataset_path = next((p for p in candidate_files if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create processed datasets")

df = pd.read_csv(dataset_path)
target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Using dataset:", dataset_path.name)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Using dataset: desharnais_processed.csv
Train shape: (64, 12) Test shape: (17, 12)


In [2]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
}

try:
    from xgboost import XGBRegressor
    models["xgboost"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )
except ImportError:
    print("xgboost is not installed. Install with: pip install xgboost")

In [3]:
metrics_frames = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    row = evaluate_predictions(name, y_test, preds)
    metrics_frames.append(row)

baseline_metrics = pd.concat(metrics_frames, ignore_index=True)
baseline_metrics = baseline_metrics.sort_values("rmse")
baseline_metrics

,model,mae,rmse
0,linear_regression,1435.054687,1943.914123
2,xgboost,1693.692017,2245.187743
1,random_forest,1755.944118,2283.387200


In [4]:
metrics_path = save_metrics(
    baseline_metrics,
    file_name="baseline_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved baseline metrics to:", metrics_path)

Saved baseline metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\baseline_metrics.csv
